In [1]:
# Kruskall test here (H test or nonparametric ANOVA)

In [2]:
import pandas as pd
import numpy as np
import scipy as sp
import sklearn
import importlib
import databox as db
import joblib
from hashlib import md5
# import sample_group_stats as sgst
from sklearn.mixture import BayesianGaussianMixture
from sklearn.decomposition import FastICA
import extools as ex
import scipy.stats as st
import scipy.special as sp

OK -- read the ontology db (19571, 4)
Restored 303734 tags and 191240 tokens. Run update_corpus_tagdata() to update.
-- loaded the skipgram model tank/fasttext_8.model --


In [3]:


cedf = pd.read_parquet('data/q_sep/CE-ICA_data_8-dim.pq')
wgdf = pd.read_parquet('data/q_sep/WG-ICA_data_8-dim.pq')

cedf

0         1         2  \
category   token_id                                                    
topic      Eur-987c_25_vole-3          -0.666552  0.022111  1.186840   
taxongroup Sci-501e_7_squirrel-5        0.437882 -0.811388  0.390039   
topic      Aur-441a_64_jellyfish-30    -0.026720 -0.699322 -0.699815   
           Cli-c809_26_animal-15       -0.409737 -0.535458  0.564966   
           Blab943_11_bird-4           -0.917884 -0.693017 -0.027334   
...                                          ...       ...       ...   
           Sem-88a0_194_Hymenoptera-27  0.037199 -0.702670 -1.493168   
taxongroup Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Rep-aa64_32_animals-26      -0.711836  0.719764 -0.809127   
           Pap-75c9_71_Papilio-42       1.411378 -0.681664  1.200986   

                                               3         4         5  \
category   token_id                                                    
topic      Eur-987c_25_vole-3           0.275075 -0.343453 -0.190398   
taxongroup Sci-501e_7_squirrel-5        0.792916 -0.113640 -1.519081   
topic      Aur-441a_64_jellyfish-30    -0.574663 -0.736788  0.411543   
           Cli-c809_26_animal-15        1.442520  0.227825 -0.542453   
           Blab943_11_bird-4           -0.217798  0.241200 -0.264970   
...                                          ...       ...       ...   
           Sem-88a0_194_Hymenoptera-27 -0.015171  4.126123  0.032796   
taxongroup Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Rep-aa64_32_animals-26       0.375698 -0.843446  0.428591   
           Pap-75c9_71_Papilio-42       1.385699 -2.137870  0.232634   

                                               6         7  
category   token_id                                         
topic      Eur-987c_25_vole-3          -0.318431  0.340504  
taxongroup Sci-501e_7_squirrel-5       -0.657930  1.530297  
topic      Aur-441a_64_jellyfish-30    -0.603104  0.550438  
           Cli-c809_26_animal-15        0.014104 -0.415746  
           Blab943_11_bird-4           -0.315390  1.291464  
...                                          ...       ...  
           Sem-88a0_194_Hymenoptera-27 -0.065156  0.388183  
taxongroup Rep-aa64_32_animals-26      -0.062365  0.129428  
           Rep-aa64_32_animals-26      -0.062365  0.129428  
           Rep-aa64_32_animals-26      -0.062365  0.129428  
           Pap-75c9_71_Papilio-42       0.051112  0.597190  

[54011 rows x 8 columns]

In [4]:
categories = cedf.index.levels[0]

In [5]:
# FIRST COMPARE H test and U test in 200 row slices or less: 
# THEN DECISION: Choose H-test
# both seem to give similar p-values, but U test is recommended for N<30 rows

htests = []
utests = []
for (m,df) in [('ce',cedf),('wg',wgdf)]:
    for cat1 in categories:
        for cat2 in categories:
            for di in df.columns:
                    
                    d,e = df.loc[cat1].iloc[0:200][di],df.loc[cat2].iloc[0:200][di]                
                    
                    utests.append((m,cat1,cat2,di,int(st.mannwhitneyu(d,e).pvalue*10000)/10000))                
                    htests.append((m,cat1,cat2,di,int(st.kruskal(d,e).pvalue*10000)/10000))
u_df = pd.DataFrame(utests,columns=['dataset','cat1','cat2','dimid','pvalue']).set_index('dataset')
h_df = pd.DataFrame(htests,columns=['dataset','cat1','cat2','dimid','pvalue']).set_index('dataset')

u_df.sample(4),h_df.sample(4)


    

(               cat1        cat2  dimid  pvalue
 dataset                                       
 ce         location    location      6  1.0000
 ce       taxongroup  taxongroup      5  1.0000
 ce       adaptation  taxongroup      1  0.4567
 wg         location  adaptation      5  0.4457,
                cat1        cat2  dimid  pvalue
 dataset                                       
 wg            topic       topic      0  1.0000
 wg       adaptation    location      3  0.0002
 wg            topic  adaptation      0  0.0357
 ce       taxongroup    location      4  0.1906)

In [37]:
# First the CE data

df =  h_df.loc['ce'].copy()
df['diff_axis']=df.pvalue<0.05
htest_ce = pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum) # H test CE
htest_ce


cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,2,1,2
location,2,0,2,1
taxongroup,1,2,0,2
topic,2,1,2,0


In [7]:
df =  u_df.loc['ce'].copy()
df['diff_axis']=df.pvalue<0.05
pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum) # U test CE

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,2,1,2
location,2,0,2,1
taxongroup,1,2,0,2
topic,2,1,2,0


In [8]:
# Word-grams:

In [38]:
df =  h_df.loc['wg'].copy()
df['diff_axis']=df.pvalue<0.05
htest_wg = pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum)  # H test WG
htest_wg


cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,6,7,6
location,6,0,5,4
taxongroup,7,5,0,2
topic,6,4,2,0


In [10]:
df =  u_df.loc['wg'].copy()
df['diff_axis']=df.pvalue<0.05
pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum) # U test WG

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,6,7,6
location,6,0,5,4
taxongroup,7,5,0,2
topic,6,4,2,0


In [65]:
# H test for complete data series (both CE and WG data)

krdata = []
for (m,df) in [('ce',cedf),('wg',wgdf)]:
    for cat1 in categories:
        for cat2 in categories:
            for di in df.columns:
                h = st.kruskal(df.loc[cat1][di],df.loc[cat2][di])
                kd = (m,cat1,cat2,di,int(h.pvalue*10000)/10000)
                #print(kd)
                krdata.append(kd)


kr_df = pd.DataFrame(krdata,columns=['dataset','cat1','cat2','dimid','pvalue']).set_index('dataset')


In [66]:
df =  kr_df.loc['ce'].copy()
df['diff_axis']=df.pvalue<0.05
ce_kruskall_ax = pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum)
ce_kruskall_ax

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,6,7,6
location,6,0,5,6
taxongroup,7,5,0,8
topic,6,6,8,0


In [67]:
pd.crosstab(df.cat1,df.cat2,values=df.pvalue,aggfunc=np.median)

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,1.00000,0.01815,0.0000,0.01810
location,0.01815,1.00000,0.0024,0.00315
taxongroup,0.00000,0.00240,1.0000,0.00000
topic,0.01810,0.00315,0.0000,1.00000


In [68]:
# Separating axis test with Syntactic Context Encodings

# number of stat. sign. separating axes with H-test

print(ce_kruskall_ax.to_latex())

\begin{tabular}{lrrrr}
\toprule
cat2 & adaptation & location & taxongroup & topic \\
cat1 &  &  &  &  \\
\midrule
adaptation & 0 & 6 & 7 & 6 \\
location & 6 & 0 & 5 & 6 \\
taxongroup & 7 & 5 & 0 & 8 \\
topic & 6 & 6 & 8 & 0 \\
\bottomrule
\end{tabular}



In [69]:
# For your interest, here are the Wordgram tables

In [70]:
df =  kr_df.loc['wg'].copy()
df['diff_axis']=df.pvalue<0.05
wg_kruskall_ax =pd.crosstab(df.cat1,df.cat2,values=df.diff_axis,aggfunc=np.sum)
wg_kruskall_ax

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,0,8,8,8
location,8,0,8,7
taxongroup,8,8,0,7
topic,8,7,7,0


In [71]:
pd.crosstab(df.cat1,df.cat2,values=df.pvalue,aggfunc=np.median)

cat2,adaptation,location,taxongroup,topic
cat1,,,,
adaptation,1.0,0.0,0.0,0.0
location,0.0,1.0,0.0,0.0
taxongroup,0.0,0.0,1.0,0.0
topic,0.0,0.0,0.0,1.0


In [72]:
# number of stat. sign. separating axes using WG, with H-test, as Reference data set

print(wg_kruskall_ax.to_latex())

\begin{tabular}{lrrrr}
\toprule
cat2 & adaptation & location & taxongroup & topic \\
cat1 &  &  &  &  \\
\midrule
adaptation & 0 & 8 & 8 & 8 \\
location & 8 & 0 & 8 & 7 \\
taxongroup & 8 & 8 & 0 & 7 \\
topic & 8 & 7 & 7 & 0 \\
\bottomrule
\end{tabular}



In [73]:
cedf.reset_index().category.value_counts() # Add N counts to table + D=8

category
topic         39533
taxongroup     8361
adaptation     3578
location       2539
Name: count, dtype: int64

In [76]:
print(", ".join(["N_{\mathrm{%s}}=%d"%(x,y) for x,y in cedf.reset_index().category.value_counts().items()]))

N_{\mathrm{topic}}=39533, N_{\mathrm{taxongroup}}=8361, N_{\mathrm{adaptation}}=3578, N_{\mathrm{location}}=2539
